In [1]:
import pandas as pd
import numpy as np
from pgmpy.models import BayesianModel
from pgmpy.models import BayesianNetwork
from pgmpy.inference import VariableElimination, ApproxInference, BeliefPropagation
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.estimators import BayesianEstimator
from pgmpy.estimators import HillClimbSearch
from pgmpy.estimators import BDeuScore, K2Score, BicScore
from pgmpy.metrics import structure_score
from pgmpy.utils import get_example_model
from pgmpy.estimators import ScoreCache
from pgmpy.inference.CausalInference import CausalInference
import networkx as nx
import bnlearn as bn
import itertools
import math
import networkx as nx
import matplotlib.pyplot as plt

In [3]:
from monte_carlo_sdp import *
from utils import *
from same_decision_probability_calculation import *

In [2]:
import networkx as nx
import numpy as np
from pgmpy.models import BayesianNetwork
from pgmpy.factors.discrete import TabularCPD

def generate_controlled_binary_bn(num_nodes=20, edge_prob=0.2, cpd_alpha=1.0):
    """
    Generates a random binary Bayesian Network with controlled parameters.
    cpd_alpha < 1.0 creates deterministic/rigid networks (closer to 0 or 1).
    cpd_alpha > 1.0 creates fuzzy/uncertain networks (closer to 0.5).
    """
    # 1. Generate a random DAG using NetworkX
    # We enforce a topological order (i < j) to guarantee no cycles
    G = nx.DiGraph()
    G.add_nodes_from(range(num_nodes))
    for i in range(num_nodes):
        for j in range(i + 1, num_nodes):
            if np.random.rand() < edge_prob:
                G.add_edge(i, j)
                
    # Convert to pgmpy format (using string names to avoid int indexing bugs!)
    edges = [(f"V{u}", f"V{v}") for u, v in G.edges()]
    bn = BayesianNetwork(edges)
    
    # Add isolated nodes if any exist
    for i in range(num_nodes):
        if f"V{i}" not in bn.nodes():
            bn.add_node(f"V{i}")

    # 2. Populate CPDs using the controlled Beta distribution
    for node in bn.nodes():
        parents = list(bn.get_parents(node))
        num_parents = len(parents)
        num_parent_states = 2 ** num_parents
        
        # Draw probabilities from Beta(alpha, alpha)
        # E.g., if alpha=0.1, p_true will mostly be ~0.99 or ~0.01
        p_true = np.random.beta(cpd_alpha, cpd_alpha, size=num_parent_states)
        p_false = 1 - p_true
        
        # TabularCPD expects values as a list of lists: [[P(False)], [P(True)]]
        cpd_values = [p_false.tolist(), p_true.tolist()]
        
        cpd = TabularCPD(
            variable=node,
            variable_card=2,
            values=cpd_values,
            evidence=parents if num_parents > 0 else None,
            evidence_card=[2] * num_parents if num_parents > 0 else None,
            state_names={node: [0, 1], **{p: [0, 1] for p in parents}}
        )
        bn.add_cpds(cpd)
        
    assert bn.check_model()
    return bn

In [ ]:
def generate_patient_evidence(model, target, min_obs=5, max_obs=15):
    all_variables = model.nodes()
    possible_evidence_vars = [v for v in all_variables if v != target]
    
    # Randomly decide how many variables are known for this patient
    #num_observed = random.randint(min_obs, max_obs)
    num_observed = 20
    observed_vars = random.sample(possible_evidence_vars, num_observed)
    
    evidence = {}
    for var in observed_vars:
        # Fetch valid states for this specific variable from the BN
        valid_states = model.get_cpds(var).state_names[var]
        evidence[var] = random.choice(valid_states)
        
    return evidence

In [ ]:
import time
import random

results = []
# Define the experimental grid
network_sizes = [15, 20, 25] 
connectivities = [0.1, 0.3] 
alphas = [0.1, 1.0, 5.0] # 0.1 = Rigid, 1.0 = Uniform, 5.0 = Fuzzy
evidence_ratios = [0.2, 0.5]

for N in network_sizes:
    for edge_prob in connectivities:
        for alpha in alphas:
            # 1. Generate the network
            bn = generate_controlled_binary_bn(N, edge_prob, alpha)
            nodes = list(bn.nodes())
            
            for e_ratio in evidence_ratios:
                # 2. Pick a random target and evidence
                target = random.choice(nodes)
                num_evidence = int(N * e_ratio)
                evidence_vars = random.sample([n for n in nodes if n != target], num_evidence)
                
                # 3. Generate a valid patient
                patient = generate_patient_evidence(bn, target) # Use your batch sampler idea here!
                
                # 4. Time and Run EXACT SDP
                start = time.time()
                exact_sdp = fast_broadcast_sdp(...) 
                exact_time = time.time() - start
                
                # 5. Time and Run MCMC SDP
                start = time.time()
                mcmc_sdp = mcmc_sdp_estimation(...)
                mcmc_time = time.time() - start
                
                # 6. Record
                results.append({
                    'N': N, 'Edge_Prob': edge_prob, 'Alpha': alpha, 'E_Ratio': e_ratio,
                    'Exact_SDP': exact_sdp, 'MCMC_SDP': mcmc_sdp,
                    'Error': abs(exact_sdp - mcmc_sdp),
                    'Exact_Time': exact_time, 'MCMC_Time': mcmc_time
                })